<a href="https://colab.research.google.com/github/BHOOMI-SONI/EDA/blob/main/Titanic_Preprocessing_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 Titanic Dataset — Data Preprocessing & Exploratory Data Analysis

**Name:** Bhoomi  
**Course:** BCA, MIT World Peace University  
**Objective:** Clean the Titanic dataset and explore it statistically and visually to understand what factors were associated with passenger survival.

---

### 📑 Table of Contents
1. Import Libraries & Load Data
2. Understanding the Dataset
3. Handling Missing Values
4. Handling Outliers
5. Encoding Categorical Variables
6. Feature Scaling & Normalization
7. Statistical Summary
8. Univariate Analysis (Graphical & Non-Graphical)
9. Multivariate Analysis (Graphical & Non-Graphical)
10. Conclusion

---
## 1. Import Libraries & Load Data

In [ ]:
# Core libraries for data handling and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look clean and readable
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

In [ ]:
# Load the Titanic dataset (built into seaborn, no file upload needed)
df = sns.load_dataset('titanic')

print('Dataset loaded successfully ✅')
print('Shape:', df.shape)
df.head()

---
## 2. Understanding the Dataset
Before touching the data, we look at its structure: column names, data types, and a quick statistical overview.

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe()

In [ ]:
# Summary of categorical columns
df.describe(include='object')

**Column meanings (quick reference):**
- `survived` → 0 = No, 1 = Yes (this is our **outcome/target variable**)
- `pclass` → Passenger class (1 = 1st, 2 = 2nd, 3 = 3rd)
- `sibsp` → Number of siblings/spouses aboard
- `parch` → Number of parents/children aboard
- `fare` → Ticket fare paid
- `embarked` → Port of embarkation (C, Q, S)

---
## 3. Handling Missing Values

In [ ]:
# Check how many values are missing in each column
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_percent.round(2)})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

In [ ]:
# Visualize missing values
plt.figure(figsize=(8, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Map (yellow = missing)')
plt.show()

**Strategy:**
- `age` → fill with **median** (numeric, robust to outliers)
- `embarked` → fill with **mode** (categorical, most frequent port)
- `deck` → **drop column** (over 75% missing, not worth imputing)
- `embark_town` → duplicate info of `embarked`, safe to drop

In [ ]:
# Fill missing numeric values with the median
df['age'] = df['age'].fillna(df['age'].median())

# Fill missing categorical values with the mode
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Drop columns that are mostly empty or redundant
df = df.drop(columns=['deck', 'embark_town'])

# Confirm the dataset is now clean
print('Remaining missing values:')
df.isnull().sum()

---
## 4. Handling Outliers
We check the `fare` column for outliers using the **IQR (Interquartile Range) method**.

In [ ]:
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['fare'] < lower_bound) | (df['fare'] > upper_bound)]
print(f'Number of outliers in fare: {len(outliers)} out of {len(df)} rows')
print(f'Normal fare range: {lower_bound:.2f} to {upper_bound:.2f}')

In [ ]:
# Visualize outliers with a boxplot
plt.figure(figsize=(8, 3))
sns.boxplot(x=df['fare'], color='salmon')
plt.title('Fare Distribution — dots beyond whiskers are outliers')
plt.show()

*Note: We keep these outliers rather than removing them — high fares genuinely belonged to 1st class passengers, so removing them would lose real information, not noise.*

---
## 5. Encoding Categorical Variables
Machine learning models need numbers, not text, so we convert categorical columns.

In [ ]:
# Label encoding for a 2-category column (sex)
df['sex_encoded'] = df['sex'].map({'male': 0, 'female': 1})

# One-hot encoding for a 3-category column (embarked)
df = pd.get_dummies(df, columns=['embarked'], prefix='embarked')

df.head()

---
## 6. Feature Scaling & Normalization
Scaling brings numeric columns onto a comparable range — useful before feeding data into most ML models.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Standardization -> mean = 0, std = 1
scaler = StandardScaler()
df['fare_standardized'] = scaler.fit_transform(df[['fare']])

# Normalization -> squashes values into a 0-1 range
minmax = MinMaxScaler()
df['fare_normalized'] = minmax.fit_transform(df[['fare']])

df[['fare', 'fare_standardized', 'fare_normalized']].head()

---
## 7. Statistical Summary
Key non-graphical statistics that summarize the numeric columns.

In [ ]:
# Measures of central tendency
print('--- AGE ---')
print('Mean  :', round(df['age'].mean(), 2))
print('Median:', round(df['age'].median(), 2))
print('Mode  :', df['age'].mode()[0])

print('\n--- FARE ---')
print('Mean  :', round(df['fare'].mean(), 2))
print('Median:', round(df['fare'].median(), 2))
print('Mode  :', df['fare'].mode()[0])

In [ ]:
# Measures of dispersion
print('--- AGE ---')
print('Std Dev :', round(df['age'].std(), 2))
print('Variance:', round(df['age'].var(), 2))
print('Range   :', df['age'].max() - df['age'].min())

print('\n--- FARE ---')
print('Std Dev :', round(df['fare'].std(), 2))
print('Variance:', round(df['fare'].var(), 2))
print('Range   :', round(df['fare'].max() - df['fare'].min(), 2))

In [ ]:
# Overall survival rate
survival_rate = df['survived'].mean() * 100
print(f'Overall survival rate: {survival_rate:.1f}%')

---
## 8. Univariate Analysis
### 8a. Non-Graphical — Frequency Tables

In [ ]:
# Frequency distribution of passenger class
df['pclass'].value_counts()

In [ ]:
# Frequency distribution of sex
df['sex'].value_counts()

### 8b. Graphical — Histograms & Boxplots

In [ ]:
# Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['age'], kde=True, bins=30, color='steelblue')
plt.title('Age Distribution of Passengers')
plt.xlabel('Age')
plt.show()

In [ ]:
# Fare distribution
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['fare'], color='lightgreen')
plt.title('Fare Boxplot')
plt.show()

In [ ]:
# Survival counts
plt.figure(figsize=(6, 5))
sns.countplot(x='survived', data=df, hue='survived', legend=False, palette='Set2')
plt.xticks([0, 1], ['Did Not Survive', 'Survived'])
plt.title('Survival Counts')
plt.show()

---
## 9. Multivariate Analysis
### 9a. Non-Graphical — Cross-tabulation & Correlation

In [ ]:
# Cross-tabulation: sex vs survival
pd.crosstab(df['sex'], df['survived'], margins=True)

In [ ]:
# Cross-tabulation: passenger class vs survival
pd.crosstab(df['pclass'], df['survived'], margins=True)

In [ ]:
# Correlation matrix of numeric columns
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df[numeric_cols].corr()
corr_matrix

### 9b. Graphical — Scatter Plot & Heat Map

In [ ]:
# Scatter plot: Age vs Fare, colored by survival
plt.figure(figsize=(8, 5))
sns.scatterplot(x='age', y='fare', hue='survived', data=df, palette='coolwarm', alpha=0.7)
plt.title('Age vs Fare, Colored by Survival')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Survival rate by class and sex combined
sns.catplot(x='pclass', y='survived', hue='sex', data=df, kind='bar', palette='Set1', height=5, aspect=1.2)
plt.title('Survival Rate by Class and Sex')
plt.show()

---
## 10. Conclusion

**Data Cleaning:**
- Filled missing `age` with median and `embarked` with mode; dropped `deck` (too sparse) and `embark_town` (redundant).
- Detected fare outliers using the IQR method but retained them as genuine 1st-class fares.
- Encoded `sex` and `embarked` into numeric form and scaled `fare` for future modeling use.

**Key Insights:**
- Overall survival rate was about **38%**.
- **Sex** was the strongest single factor — women survived at a far higher rate than men.
- **Passenger class** mattered greatly — 1st class passengers survived much more often than 3rd class.
- **Age and fare** show mild correlation with survival, but sex and class dominate.
- Combining class and sex shows 1st-class women had the highest survival rate, while 3rd-class men had the lowest.

*This notebook demonstrates the full preprocessing → EDA → statistical analysis pipeline as covered in Unit 2.*